In [0]:
%sql
use catalog test;
use schema sql;

In [0]:
%sql
-- Set-up the tables
DROP TABLE IF EXISTS sales;
CREATE TABLE sales
(
    company    VARCHAR(20),
    year       INTEGER,
    quarter    CHAR(2),
    volume     INTEGER
);
INSERT INTO sales (company, year, quarter, volume)
VALUES
    ('BYD', 2022,'Q1',143223),
    ('BYD', 2022,'Q2',180296),
    ('BYD', 2022,'Q3',258610),
    ('BYD', 2022,'Q4',329011),
    ('BYD', 2023,'Q1',264647),
    ('BYD', 2023,'Q2',352163),
    ('BYD', 2023,'Q3',431603),
    ('BYD', 2023,'Q4',526409),    
    ('Tesla', 2022,'Q1',310048),
    ('Tesla', 2022,'Q2',254695),
    ('Tesla', 2022,'Q3',343830),
    ('Tesla', 2022,'Q4',405278),
    ('Tesla', 2023,'Q1',422875),
    ('Tesla', 2023,'Q2',466140),
    ('Tesla', 2023,'Q3',435059),
    ('Tesla', 2023,'Q4',484507);	

SELECT * FROM sales;

company,year,quarter,volume
BYD,2022,Q1,143223
BYD,2022,Q2,180296
BYD,2022,Q3,258610
BYD,2022,Q4,329011
BYD,2023,Q1,264647
BYD,2023,Q2,352163
BYD,2023,Q3,431603
BYD,2023,Q4,526409
Tesla,2022,Q1,310048
Tesla,2022,Q2,254695


In [0]:
# Create DataFrame using schema inference (Spark Session)
# Spark automatically determines the data types based on the input data instead of manually injecting the schema.

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = [
    ("BYD", 2022, "Q1", 143223),
    ("BYD", 2022, "Q2", 180296),
    ("BYD", 2022, "Q3", 258610),
    ("BYD", 2022, "Q4", 329011),
    ("BYD", 2023, "Q1", 264647),
    ("BYD", 2023, "Q2", 352163),
    ("BYD", 2023, "Q3", 431603),
    ("BYD", 2023, "Q4", 526409),
    ("Tesla", 2022, "Q1", 310048),
    ("Tesla", 2022, "Q2", 254695),
    ("Tesla", 2022, "Q3", 343830),
    ("Tesla", 2022, "Q4", 405278),
    ("Tesla", 2023, "Q1", 422875),
    ("Tesla", 2023, "Q2", 466140),
    ("Tesla", 2023, "Q3", 435059),
    ("Tesla", 2023, "Q4", 484507)
]
columns = ["company", "year", "quarter", "volume"]
sales = spark.createDataFrame(data, columns)
sales.display()
sales.printSchema()

company,year,quarter,volume
BYD,2022,Q1,143223
BYD,2022,Q2,180296
BYD,2022,Q3,258610
BYD,2022,Q4,329011
BYD,2023,Q1,264647
BYD,2023,Q2,352163
BYD,2023,Q3,431603
BYD,2023,Q4,526409
Tesla,2022,Q1,310048
Tesla,2022,Q2,254695


root
 |-- company: string (nullable = true)
 |-- year: long (nullable = true)
 |-- quarter: string (nullable = true)
 |-- volume: long (nullable = true)



In [0]:
# Create DataFrame using an explicit schema. 
# Where we define the data types & make the schema 
# Recommended for production ETL pipelines and large datasets.

from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("company", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("quarter", StringType(), True),
    StructField("volume", IntegerType(), True)
])
sales_df = spark.createDataFrame(data, schema)
display(sales_df)
sales_df.printSchema()
sales_df.write.mode("overwrite").saveAsTable("test.pyspark.sales")

company,year,quarter,volume
BYD,2022,Q1,143223
BYD,2022,Q2,180296
BYD,2022,Q3,258610
BYD,2022,Q4,329011
BYD,2023,Q1,264647
BYD,2023,Q2,352163
BYD,2023,Q3,431603
BYD,2023,Q4,526409
Tesla,2022,Q1,310048
Tesla,2022,Q2,254695


root
 |-- company: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- volume: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql.window import Window

sales_df.withColumn(
    "sales_per_year",
    sum(col("volume")).over(Window.partitionBy(col("company"), col("year")))
).display()

company,year,quarter,volume,sales_per_year
BYD,2022,Q1,143223,911140
BYD,2022,Q2,180296,911140
BYD,2022,Q3,258610,911140
BYD,2022,Q4,329011,911140
BYD,2023,Q1,264647,1574822
BYD,2023,Q2,352163,1574822
BYD,2023,Q3,431603,1574822
BYD,2023,Q4,526409,1574822
Tesla,2022,Q1,310048,1313851
Tesla,2022,Q2,254695,1313851


In [0]:
%sql
select 
    *,
    sum(volume) over (partition by company, year)
    as total_volume
from sales

company,year,quarter,volume,total_volume
BYD,2022,Q1,143223,911140
BYD,2022,Q2,180296,911140
BYD,2022,Q3,258610,911140
BYD,2022,Q4,329011,911140
BYD,2023,Q1,264647,1574822
BYD,2023,Q2,352163,1574822
BYD,2023,Q3,431603,1574822
BYD,2023,Q4,526409,1574822
Tesla,2022,Q1,310048,1313851
Tesla,2022,Q2,254695,1313851


##### LAG function

###### LAG(column, offset, default) --> returns the value from a previous row based on the ORDER BY sequence. 
- 'offset' (default = 1) --> specifies how many rows back to look.
- 'default' --> We can mention it manually to replace NULL when no previous row exists. 

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) resets the calculation for each group. Commonly used for MoM, YoY, QoQ growth and comparing current values with previous records.

In [0]:
%sql
select
    *,
    lag(volume, 1, 0) over(order by year, quarter) as prev_volume,
    volume + (lag(volume, 1, 0) over(order by year, quarter)) as growth
from sales
where company = "Tesla";

company,year,quarter,volume,prev_volume,growth
Tesla,2022,Q1,310048,0,310048
Tesla,2022,Q2,254695,310048,564743
Tesla,2022,Q3,343830,254695,598525
Tesla,2022,Q4,405278,343830,749108
Tesla,2023,Q1,422875,405278,828153
Tesla,2023,Q2,466140,422875,889015
Tesla,2023,Q3,435059,466140,901199
Tesla,2023,Q4,484507,435059,919566


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sales_df.withColumn(
    "prev_quarter_sales",
    lag(col("volume"),1,0).over(Window.orderBy(col("year"), col("quarter")))
).withColumn(
    "growth",
    (col("volume") + col("prev_quarter_sales"))
).filter(
    col("company") == "Tesla"
).display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


company,year,quarter,volume,prev_quarter_sales,growth
Tesla,2022,Q1,310048,143223,453271
Tesla,2022,Q2,254695,180296,434991
Tesla,2022,Q3,343830,258610,602440
Tesla,2022,Q4,405278,329011,734289
Tesla,2023,Q1,422875,264647,687522
Tesla,2023,Q2,466140,352163,818303
Tesla,2023,Q3,435059,431603,866662
Tesla,2023,Q4,484507,526409,1010916


##### LEAD function

###### LEAD(column, offset, default) --> returns the value from a next row based on the ORDER BY sequence.
- 'offset' (default = 1) --> specifies how many rows ahead to look.
- 'default' --> We can mention it manually to replace NULL when no next row exists.

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) resets the calculation for each group. Commonly used to compare the current row with future records, calculate forward differences, and identify the next event or transaction.

In [0]:
%sql
select
    *,
    lead(volume,1,0) over(partition by company order by year, quarter) as next_volume,
    volume + (lead(volume,1,0) over(partition by company order by year, quarter)) as growth
from sales;

company,year,quarter,volume,next_volume,growth
BYD,2022,Q1,143223,180296,323519
BYD,2022,Q2,180296,258610,438906
BYD,2022,Q3,258610,329011,587621
BYD,2022,Q4,329011,264647,593658
BYD,2023,Q1,264647,352163,616810
BYD,2023,Q2,352163,431603,783766
BYD,2023,Q3,431603,526409,958012
BYD,2023,Q4,526409,0,526409
Tesla,2022,Q1,310048,254695,564743
Tesla,2022,Q2,254695,343830,598525


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sales_df.withColumn(
    "next_volume",
    lead(col("volume"),1,0).over(
                Window.partitionBy(col("company")).orderBy(col("year"), col("quarter"))
    )
).withColumn(
    "growth",
    (col("next_volume") + col("volume"))
).display()

company,year,quarter,volume,next_volume,growth
BYD,2022,Q1,143223,180296,323519
BYD,2022,Q2,180296,258610,438906
BYD,2022,Q3,258610,329011,587621
BYD,2022,Q4,329011,264647,593658
BYD,2023,Q1,264647,352163,616810
BYD,2023,Q2,352163,431603,783766
BYD,2023,Q3,431603,526409,958012
BYD,2023,Q4,526409,0,526409
Tesla,2022,Q1,310048,254695,564743
Tesla,2022,Q2,254695,343830,598525


#### ROW_NUMBER() function

###### ROW_NUMBER() --> Assigns a unique sequential number to each row within a partition based on the ORDER BY sequence, even if there are duplicate values.

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) restarts numbering for each group. Commonly used for deduplication, selecting the latest/first record, and Top-N records.

#### RANK() function

###### RANK() --> Assigns a rank to each row within a partition based on the ORDER BY sequence. Duplicate values receive the same rank, and the next rank(s) are skipped.

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) restarts ranking for each group. Commonly used for competition ranking, leaderboard reports, and identifying Top-N records with ties.

#### DENSE_RANK() function

####### DENSE_RANK() --> Assigns a rank to each row within a partition based on the ORDER BY sequence. Duplicate values receive the same rank, but the next rank is not skipped.

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) restarts ranking for each group. Commonly used when consecutive rankings are required, such as Top-N analysis and reporting.

![image_1783494643662.png](./image_1783494643662.png "image_1783494643662.png")

In [0]:
%sql
with cte as(
    select 
        *,
        row_number() over (partition by company, year order by volume desc) as row_no
    from sales
)
select * from cte where row_no = 1;

company,year,quarter,volume,row_no
BYD,2022,Q4,329011,1
BYD,2023,Q4,526409,1
Tesla,2022,Q4,405278,1
Tesla,2023,Q4,484507,1


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sales_df.withColumn(
    "row_no",
    row_number().over(Window.partitionBy(col("company"), col("year")).orderBy(desc(col("volume"))))
).filter(
    col("row_no") == 1
).display()

company,year,quarter,volume,row_no
BYD,2022,Q4,329011,1
BYD,2023,Q4,526409,1
Tesla,2022,Q4,405278,1
Tesla,2023,Q4,484507,1


##### NTILE() function

###### NTILE(n) --> Divides the rows into 'n' nearly equal groups (buckets) based on the ORDER BY sequence and assigns a bucket number starting from 1.

###### It must be used with OVER(ORDER BY ...), and PARTITION BY (optional) creates separate buckets for each group. Commonly used for quartiles, percentiles, customer segmentation, salary bands, and data distribution analysis.

In [0]:
%sql
select 
    *,
    ntile(2) over(partition by company, year order by volume desc) as parts  
from sales;

company,year,quarter,volume,parts
BYD,2022,Q4,329011,1
BYD,2022,Q3,258610,1
BYD,2022,Q2,180296,2
BYD,2022,Q1,143223,2
BYD,2023,Q4,526409,1
BYD,2023,Q3,431603,1
BYD,2023,Q2,352163,2
BYD,2023,Q1,264647,2
Tesla,2022,Q4,405278,1
Tesla,2022,Q3,343830,1


##### Window Frames

###### Window Frames define which rows around the current row should be included in a window calculation. They are specified inside the OVER() clause after ORDER BY using :
######  **`ROWS BETWEEN lower_boundary AND upper_boundary`**

###### The lower and upper boundaries can be:
- CURRENT ROW --> The current row.
- n PRECEDING --> n rows before the current row.
- n FOLLOWING --> n rows after the current row.
- UNBOUNDED PRECEDING --> The first row of the partition.
- UNBOUNDED FOLLOWING --> The last row of the partition.

###### Window Frames are commonly used with aggregate window functions such as SUM(), AVG(), MIN(), MAX(), and COUNT() to calculate running totals, moving averages, rolling aggregations, and cumulative calculations.

In [0]:
%sql
---- Find the total sales on half yearly basis ----

-- select 
--     *,
--     sum(volume) over(
--         partition by(company)
--         order by (year, quarter)
--         rows between current row and 1 following
--     ) as half_yearly_sales
-- from sales;

with cte as(
    select
        *,
        lead(volume,1,0) over (
            partition by company
            order by year, quarter
        ) as next_quarter_volume
    from sales
)
select
    *,
    volume + next_quarter_volume as half_yearly_sales
from cte;

company,year,quarter,volume,next_quarter_volume,half_yearly_sales
BYD,2022,Q1,143223,180296,323519
BYD,2022,Q2,180296,258610,438906
BYD,2022,Q3,258610,329011,587621
BYD,2022,Q4,329011,264647,593658
BYD,2023,Q1,264647,352163,616810
BYD,2023,Q2,352163,431603,783766
BYD,2023,Q3,431603,526409,958012
BYD,2023,Q4,526409,0,526409
Tesla,2022,Q1,310048,254695,564743
Tesla,2022,Q2,254695,343830,598525


#### WindowFrames in PYSPARK
###### rowsBetween(start, end) defines the window frame relative to the current row. Use negative values for preceding rows and positive values for following rows.
###### Example: rowsBetween(Window.currentRow, 1) includes the current row and the next row.

![image_1783581404504.png](./image_1783581404504.png "image_1783581404504.png")

In [0]:
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

sales_df.withColumn(
    "half_yearly_sales",
    sum("volume").over(
        Window.partitionBy("company")
              .orderBy("year", "quarter")
              .rowsBetween(Window.currentRow, 1)
    )
).withColumn(
    "yearly_sales",
    sum("volume").over(
        Window.partitionBy("company")
              .orderBy("year", "quarter")
              .rowsBetween(Window.currentRow, Window.unboundedFollowing)
    )
).display()

company,year,quarter,volume,half_yearly_sales,yearly_sales
BYD,2022,Q1,143223,323519,2485962
BYD,2022,Q2,180296,438906,2342739
BYD,2022,Q3,258610,587621,2162443
BYD,2022,Q4,329011,593658,1903833
BYD,2023,Q1,264647,616810,1574822
BYD,2023,Q2,352163,783766,1310175
BYD,2023,Q3,431603,958012,958012
BYD,2023,Q4,526409,526409,526409
Tesla,2022,Q1,310048,564743,3122432
Tesla,2022,Q2,254695,598525,2812384
